# Attributing Assistant Behavior to System Prompt Clauses - Reference Solution

Given a system prompt's clauses and the model's REAL measured behaviour
with the full prompt in effect, predict which clause is most responsible
(ranking) and whether each clause is necessary or merely correlated
(flags). The released features never include per-clause ablated
fingerprints (that would leak the answer) - only the full-prompt
fingerprint, the clause texts themselves, and their (shuffled) display
order. This solution engineers per-clause features from clause TEXT and
its relationship to the shared full-prompt fingerprint, then trains a
model to predict each clause's standardized responsibility score.


In [ ]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold

SEED = 20260920
np.random.seed(SEED)

PUB = "./dataset/public" if os.path.isdir("./dataset/public") else "./prepared/public"
rd = lambda f: pd.read_csv(os.path.join(PUB, f), keep_default_na=False)

train = rd("train.csv")
test = rd("test.csv")
print(f"train: {len(train):,} rows   test: {len(test):,} rows")
train.head(3)


## What one item looks like

Each row is one system prompt shown with a subset of probe questions, its
full-prompt behaviour fingerprint (`fp__*`), and its clauses in a
RANDOMIZED display order (`clause_ids`, `clause_texts`, pipe-delimited).
The target `ranking` reorders those same clause_ids by true measured
responsibility; `flags` gives each clause a necessary/correlated label.


In [ ]:
FP_FIELDS = [c for c in train.columns if c.startswith("fp__")]
print("fingerprint fields:", FP_FIELDS)

def parse_pipe(s):
    return s.split("|") if s else []

def parse_flags(s):
    out = {}
    for pair in parse_pipe(s):
        if ":" in pair:
            cid, flag = pair.split(":", 1)
            out[cid] = flag
    return out


## Feature engineering

Per clause, per item, build features from:
- the clause's own text (length, presence of hedge/persona/scope keywords)
- the clause's position in the DISPLAYED order (a weak, deliberately-
  shuffled signal - included so the model can learn it is NOT predictive,
  rather than to exploit it)
- the shared full-prompt fingerprint for that item (broadcast to every
  clause in the item, since it is the only behavioural signal released)

The true per-clause TARGET used for training is derived from the item's
own `ranking` column: a clause's rank position (0 = most responsible)
converted to a continuous pseudo-score, since the real standardized
ablation distance itself is never released (that is the answer key).


In [ ]:
HEDGE_KEYWORDS = re.compile(r"\\b(may|might|unsure|if unsure|hedge|uncertain)\\b", re.I)
PERSONA_KEYWORDS = re.compile(r"\\byou are\\b", re.I)
SCOPE_KEYWORDS = re.compile(r"\\b(only|never|always|must|scope|focus)\\b", re.I)
FORMAT_KEYWORDS = re.compile(r"\\b(format|bullet|json|markdown|structure)\\b", re.I)


def clause_features(clause_text, position, n_clauses, fp_row):
    feats = {
        "clause_len_words": len(clause_text.split()),
        "clause_len_chars": len(clause_text),
        "has_hedge_kw": float(bool(HEDGE_KEYWORDS.search(clause_text))),
        "has_persona_kw": float(bool(PERSONA_KEYWORDS.search(clause_text))),
        "has_scope_kw": float(bool(SCOPE_KEYWORDS.search(clause_text))),
        "has_format_kw": float(bool(FORMAT_KEYWORDS.search(clause_text))),
        "display_position": position,
        "display_position_frac": position / max(n_clauses - 1, 1),
        "n_clauses": n_clauses,
    }
    for f in FP_FIELDS:
        feats[f] = fp_row[f]
    return feats


def build_feature_rows(df, has_labels):
    rows = []
    for _, row in df.iterrows():
        clause_ids = parse_pipe(row["clause_ids"])
        clause_texts = parse_pipe(row["clause_texts"])
        n = len(clause_ids)
        fp_row = {f: row[f] for f in FP_FIELDS}

        rank_target = {}
        flag_target = {}
        if has_labels:
            ranking = parse_pipe(row["ranking"])
            flags = parse_flags(row["flags"])
            for pos, cid in enumerate(ranking):
                rank_target[cid] = 1.0 - pos / max(len(ranking) - 1, 1)
            flag_target = flags

        for pos, (cid, text) in enumerate(zip(clause_ids, clause_texts)):
            feats = clause_features(text, pos, n, fp_row)
            feats["item_id"] = row["item_id"]
            feats["clause_id"] = cid
            feats["topic"] = row["topic"]
            if has_labels:
                feats["target_score"] = rank_target.get(cid, 0.5)
                feats["target_flag"] = flag_target.get(cid, "correlated")
            rows.append(feats)
    return pd.DataFrame(rows)


train_feats = build_feature_rows(train, has_labels=True)
test_feats = build_feature_rows(test, has_labels=False)
print(f"train_feats: {len(train_feats):,} clause-rows   test_feats: {len(test_feats):,}")
train_feats.head(3)


## Training

A gradient-boosted regressor predicts each clause's continuous
responsibility pseudo-score from its own features plus the shared
full-prompt fingerprint. Grouped K-fold by `topic` avoids any
topic-level leakage during validation, mirroring the real train/test
split.


In [ ]:
FEATURE_COLS = [c for c in train_feats.columns
                if c not in ("item_id", "clause_id", "topic", "target_score", "target_flag")]

X = train_feats[FEATURE_COLS].values
y = train_feats["target_score"].values
groups = train_feats["topic"].values

gkf = GroupKFold(n_splits=5)
oof_pred = np.zeros(len(train_feats))
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    model = GradientBoostingRegressor(random_state=SEED, n_estimators=200, max_depth=3)
    model.fit(X[tr_idx], y[tr_idx])
    oof_pred[va_idx] = model.predict(X[va_idx])

model = GradientBoostingRegressor(random_state=SEED, n_estimators=200, max_depth=3)
model.fit(X, y)
print("trained on", len(train_feats), "clause-rows")


## Deriving the ranking and flags from the predicted score

Within each item, sort clauses by predicted score (descending) for the
ranking. For the necessary/correlated flag, threshold the predicted score
at its 70th percentile ACROSS THE TRAINING SET's own oof predictions (the
same percentile rule used to build the true labels - see prepare.py's
LABEL RULE), applied per-item to keep the flag rate consistent with the
label construction.


In [ ]:
NECESSARY_THRESHOLD = np.percentile(oof_pred, 70)
print(f"necessary threshold (70th pct of oof predictions): {NECESSARY_THRESHOLD:.4f}")

test_feats["pred_score"] = model.predict(test_feats[FEATURE_COLS].values)

submission_rows = []
for item_id, grp in test_feats.groupby("item_id"):
    ranked = grp.sort_values("pred_score", ascending=False)
    ranking_str = "|".join(ranked["clause_id"])
    flags_str = "|".join(
        f"{cid}:{'necessary' if score >= NECESSARY_THRESHOLD else 'correlated'}"
        for cid, score in zip(ranked["clause_id"], ranked["pred_score"])
    )
    submission_rows.append({"item_id": item_id, "ranking": ranking_str, "flags": flags_str})

submission = pd.DataFrame(submission_rows)
print(f"submission: {len(submission):,} rows")
submission.head(3)


## Write submission


In [ ]:
os.makedirs("./working", exist_ok=True)
submission.to_csv("./working/submission.csv", index=False)
print("wrote ./working/submission.csv")


## Per-topic validation breakdown

Report out-of-fold performance per topic (not just an aggregate) to check
whether the learned signal transfers, or overfits to specific training
topics' phrasing style.


In [ ]:
from collections import defaultdict

train_feats["oof_pred"] = oof_pred
per_topic_corr = {}
for topic, grp in train_feats.groupby("topic"):
    if grp["target_score"].std() > 0 and grp["oof_pred"].std() > 0:
        per_topic_corr[topic] = np.corrcoef(grp["target_score"], grp["oof_pred"])[0, 1]

for topic, corr in sorted(per_topic_corr.items(), key=lambda x: -x[1]):
    print(f"  {topic}: oof correlation = {corr:.3f}")
